# Vise Portfolio Strategy — Comparative Analysis
### UT Austin MSBA // Vise Capstone

---

## Objective

This notebook runs the **full scaled sweep**: the same engine and strategy grid as the canonical 4-portfolio study (`vise_comparative_analysis.ipynb`), applied to **every model portfolio in Models.xlsx** (158 portfolios across 20 model families), grouped by equity sleeve.
- **Portfolios**: all model portfolios from Models.xlsx (renormalized to tickers with price history)
- **6 time periods**: Bear (2007–2008), Baseline (2010–2019), Bull (2023–2024), and trailing 5, 10, and 20-year windows

Within each portfolio × period, all strategy variants are stacked:
- **TLH**: off (benchmark) vs on (10% loss threshold)
- **Calendar rebalancing**: Monthly, Quarterly, Yearly
- **Absolute threshold rebalancing**: 5%, 10%, 20%
- **Relative threshold rebalancing**: 25%, 50%

Total strategies per portfolio × period: 8 × 2 (TLH on/off) = **16 runs**, for **15,136 runs** across the full sweep. This is a directional robustness check: some model portfolios hold mutual funds absent from the ETF price data and are renormalized to their available tickers.

Each no-TLH run is the benchmark for its TLH counterpart — tracking error and information ratio measure the TLH value-add.

---

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — IMPORTS & EMBEDDED ENGINE FUNCTIONS
# Engine functions are embedded directly to avoid importing the Streamlit app
# (portfolio_returns_engine.py) which fails outside a Streamlit context.
# Only optimizer_msba_v1_engine.py is imported — it is a pure Python module.
# ══════════════════════════════════════════════════════════════════════════════

import os, sys, warnings, itertools
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats as sp_stats
from IPython.display import display

warnings.filterwarnings('ignore')

# ── Resolve paths ─────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path(os.path.abspath(''))
REPO_ROOT    = NOTEBOOK_DIR.parent
DATA_DIR     = REPO_ROOT / 'data'

sys.path.insert(0, str(REPO_ROOT))
from optimizer_msba_v1_engine import run_optimizer_simulation

# ══════════════════════════════════════════════════════════════════════════════
# EMBEDDED ENGINE FUNCTIONS  (extracted from portfolio_returns_engine.py V4)
# ══════════════════════════════════════════════════════════════════════════════

def _get_rebalance_dates(trading_dates, freq):
    """Return set of rebalance dates for a given calendar frequency."""
    dates = pd.DatetimeIndex(trading_dates)
    if len(dates) < 2:
        return set()
    if freq == 'Daily':
        return set(dates[1:])
    rebal_set = set()
    prev_month = dates[0].month
    prev_year  = dates[0].year
    prev_week  = dates[0].isocalendar()[1]
    if freq == 'Weekly':
        for dt in dates[1:]:
            iso = dt.isocalendar()
            if iso[1] != prev_week or dt.year != prev_year:
                rebal_set.add(dt); prev_week = iso[1]; prev_year = dt.year
    elif freq == 'Monthly':
        for dt in dates[1:]:
            if dt.month != prev_month or dt.year != prev_year:
                rebal_set.add(dt); prev_month = dt.month; prev_year = dt.year
    elif freq == 'Quarterly':
        for dt in dates[1:]:
            if dt.month in {1, 4, 7, 10} and (dt.month != prev_month or dt.year != prev_year):
                rebal_set.add(dt)
            if dt.month != prev_month or dt.year != prev_year:
                prev_month = dt.month; prev_year = dt.year
    elif freq == 'Yearly':
        for dt in dates[1:]:
            if dt.year != prev_year:
                rebal_set.add(dt); prev_month = dt.month; prev_year = dt.year
    else:
        raise ValueError(f'Unknown rebalance frequency: {freq}')
    return rebal_set


def build_rebalanced_series(prices_wide, target_weights, initial_capital, rebalance_freq,
                            cost_bps=12.0):
    """
    Calendar-only rebalancing engine (V3). Returns (rebal_daily, rebal_stats).

    cost_bps : total one-way transaction cost in basis points (commission + slippage + spread).
               Default 12 bps = 5 commission + 5 slippage + 2 bid-ask, matching optimizer engine.
               Pass 0.0 to disable cost deduction.

    NOTE — No look-ahead bias: day-i prices are used only to value existing positions and
    execute rebalancing trades. Target weights are fixed inputs computed before the loop.

    NOTE — Dividends excluded: this engine does not model dividend reinvestment. Portfolios
    with high-yielding constituents will show slightly lower NAV relative to the TLH engine,
    which models DRIP. This is a data-availability constraint, not a model choice.
    """
    cost_rate = cost_bps / 10_000.0
    tickers = list(target_weights.keys())
    dates   = prices_wide.index.tolist()
    n_days  = len(dates)
    if n_days == 0:
        raise ValueError('No trading dates in the filtered price data.')
    rebal_dates = _get_rebalance_dates(dates, rebalance_freq)
    shares = {tk: initial_capital * target_weights[tk] / prices_wide.loc[dates[0], tk]
              for tk in tickers}
    portfolio_values        = np.empty(n_days, dtype=np.float64)
    rebalance_count         = 0
    total_turnover_dollars  = 0.0
    total_transaction_costs = 0.0
    for i, dt in enumerate(dates):
        tv = {tk: shares[tk] * prices_wide.loc[dt, tk] for tk in tickers}
        total_value = sum(tv.values())
        portfolio_values[i] = total_value
        if dt in rebal_dates and total_value > 0:
            if any(abs(tv[tk] / total_value - target_weights[tk]) > 1e-10 for tk in tickers):
                rebalance_count += 1
                # Compute gross turnover (dollar volume of all trades this event)
                trade_turn = sum(
                    abs(target_weights[tk] * total_value / prices_wide.loc[dt, tk] - shares[tk])
                    * prices_wide.loc[dt, tk]
                    for tk in tickers
                )
                total_turnover_dollars  += trade_turn
                cost_amount              = trade_turn * cost_rate
                total_transaction_costs += cost_amount
                # Allocate net-of-cost value to target weights so cost compounds forward
                net_value = total_value - cost_amount
                for tk in tickers:
                    shares[tk] = target_weights[tk] * net_value / prices_wide.loc[dt, tk]
                portfolio_values[i] = net_value
    avg_val = np.mean(portfolio_values)
    rebal_daily = pd.DataFrame({'Portfolio Value': portfolio_values}, index=dates)
    rebal_daily.index.name = 'PRICEDATE'
    rebal_stats = {
        'rebalance_count':         rebalance_count,
        'turnover_proxy':          round(total_turnover_dollars / avg_val if avg_val > 0 else 0, 4),
        'total_turnover_dollars':  round(total_turnover_dollars, 2),
        'transaction_costs_total': round(total_transaction_costs, 2),
        'final_value':             round(portfolio_values[-1], 2),
        'total_return':            round(portfolio_values[-1] / initial_capital - 1, 6),
    }
    return rebal_daily, rebal_stats


def compute_weights(shares, prices):
    values = {tk: shares[tk] * prices[tk] for tk in shares}
    total  = sum(values.values())
    return {tk: values[tk] / total for tk in shares} if total > 0 else {tk: 0.0 for tk in shares}


def compute_drift(current_weights, target_weights, drift_mode='Absolute'):
    """
    Compute per-asset drift between current and target weights.

    Relative mode uses the symmetric log-ratio |log(w_cur / w_tgt)| instead of
    |w_cur/w_tgt - 1|. The log formula treats over- and under-weight positions
    symmetrically: doubling (2->4%) and halving (4->2%) produce equal drift values.
    """
    drift = {}
    for tk, w_tgt in target_weights.items():
        w_cur = current_weights.get(tk, 0.0)
        if drift_mode == 'Relative':
            # FIX: symmetric log-ratio — avoids asymmetry of |w/tgt - 1|
            if w_tgt >= 1e-12 and w_cur > 1e-12:
                drift[tk] = abs(np.log(w_cur / w_tgt))
            elif w_tgt >= 1e-12:
                drift[tk] = abs(np.log(1e-12 / w_tgt))
            else:
                drift[tk] = abs(w_cur)
        else:
            drift[tk] = abs(w_cur - w_tgt)
    return drift


def find_threshold_triggers(drift, tolerances):
    return [tk for tk, d in drift.items() if d > tolerances.get(tk, 0.05) + 1e-12]


def apply_rebalance_full(shares, target_weights, prices, total_value, cost_rate=0.0):
    """
    Rebalance to target weights. Returns (new_shares, gross_turnover, cost_amount).
    Gross turnover = sum of |dollar trade| across all assets (before costs).
    Net NAV after costs = total_value - cost_amount.
    """
    trade_turn = sum(
        abs(target_weights[tk] * total_value / prices[tk] - shares[tk]) * prices[tk]
        for tk in target_weights if prices.get(tk, 0) > 0
    )
    cost_amount = trade_turn * cost_rate
    net_value   = total_value - cost_amount
    new_shares  = {
        tk: target_weights[tk] * net_value / prices[tk] if prices.get(tk, 0) > 0 else 0.0
        for tk in target_weights
    }
    return new_shares, trade_turn, cost_amount


def build_threshold_rebalanced_series(
    prices_wide, target_weights, initial_capital, tolerances,
    drift_mode='Absolute', rebalance_action='Full', cooldown_days=0,
    calendar_freq=None, enable_calendar=False, enable_threshold=True,
    cost_bps=12.0,
):
    """
    Threshold (drift-band) rebalancing engine (V4).

    cost_bps : total one-way transaction cost in basis points. Default 12 bps.

    NOTE — No look-ahead bias: breach detection on day i schedules a rebalance on day i+1;
    the trade executes at day-i+1 prices (represented here by day-i+1 close, consistent with
    the rest of the simulation). Triggers are never computed from future prices.

    NOTE — Dividends excluded: see build_rebalanced_series docstring.
    """
    cost_rate = cost_bps / 10_000.0
    tickers = list(target_weights.keys())
    dates   = prices_wide.index.tolist()
    n_days  = len(dates)
    if n_days == 0:
        raise ValueError('No trading dates in the filtered price data.')
    calendar_dates = set()
    if enable_calendar and calendar_freq and calendar_freq != 'None':
        calendar_dates = _get_rebalance_dates(dates, calendar_freq)
    shares = {tk: initial_capital * target_weights[tk] / prices_wide.loc[dates[0], tk]
              for tk in tickers}
    portfolio_values = np.empty(n_days, dtype=np.float64)
    drift_history    = {tk: [] for tk in tickers}
    event_log        = []
    rebalance_count  = calendar_rebal_count = threshold_rebal_count = 0
    total_turnover          = 0.0
    total_transaction_costs = 0.0
    cooldown_remaining = 0
    pending = False; p_tickers = []; p_max_drift = 0.0
    for i, dt in enumerate(dates):
        pt = {tk: float(prices_wide.loc[dt, tk]) for tk in tickers}
        tv = sum(shares[tk] * pt[tk] for tk in tickers)
        portfolio_values[i] = tv
        cw    = compute_weights(shares, pt)
        drift = compute_drift(cw, target_weights, drift_mode)
        for tk in tickers:
            drift_history[tk].append(drift.get(tk, 0.0))
        did_rebal = False
        if pending and enable_threshold and i > 0 and cooldown_remaining <= 0 and tv > 0:
            shares, turn, cost_amt = apply_rebalance_full(shares, target_weights, pt, tv, cost_rate)
            total_turnover += turn; total_transaction_costs += cost_amt
            rebalance_count += 1; threshold_rebal_count += 1
            did_rebal = True; cooldown_remaining = cooldown_days
            tv = sum(shares[tk] * pt[tk] for tk in tickers)
            portfolio_values[i] = tv
            event_log.append({'date': dt, 'reason': 'threshold',
                               'breached_tickers': ', '.join(p_tickers),
                               'max_drift': round(p_max_drift, 6), 'turnover_dollars': round(turn, 2)})
        pending = False; p_tickers = []; p_max_drift = 0.0
        if enable_calendar and dt in calendar_dates and tv > 0:
            cw_now = compute_weights(shares, pt)
            if any(abs(cw_now.get(tk, 0) - target_weights[tk]) > 1e-10 for tk in tickers):
                shares, turn, cost_amt = apply_rebalance_full(shares, target_weights, pt, tv, cost_rate)
                total_turnover += turn; total_transaction_costs += cost_amt
                calendar_rebal_count += 1
                if not did_rebal: rebalance_count += 1
                tv = sum(shares[tk] * pt[tk] for tk in tickers)
                portfolio_values[i] = tv
                event_log.append({'date': dt, 'reason': 'calendar', 'breached_tickers': '',
                                   'max_drift': round(max(drift.values(), default=0), 6),
                                   'turnover_dollars': round(turn, 2)})
        if enable_threshold and i < n_days - 1:
            cw_post  = compute_weights(shares, pt)
            dp       = compute_drift(cw_post, target_weights, drift_mode)
            breached = find_threshold_triggers(dp, tolerances)
            if breached and cooldown_remaining <= 0:
                pending = True; p_tickers = breached
                p_max_drift = max(dp[tk] for tk in breached)
        if cooldown_remaining > 0: cooldown_remaining -= 1
    avg_val    = np.mean(portfolio_values)
    rebal_daily = pd.DataFrame({'Portfolio Value': portfolio_values}, index=dates)
    rebal_daily.index.name = 'PRICEDATE'
    rebal_stats = {
        'rebalance_count':         rebalance_count,
        'calendar_rebal_count':    calendar_rebal_count,
        'threshold_rebal_count':   threshold_rebal_count,
        'turnover_proxy':          round(total_turnover / avg_val if avg_val > 0 else 0, 4),
        'total_turnover_dollars':  round(total_turnover, 2),
        'transaction_costs_total': round(total_transaction_costs, 2),
        'final_value':             round(portfolio_values[-1], 2),
        'total_return':            round(portfolio_values[-1] / initial_capital - 1, 6),
    }
    event_log_df = pd.DataFrame(event_log) if event_log else \
        pd.DataFrame(columns=['date', 'reason', 'breached_tickers', 'max_drift', 'turnover_dollars'])
    return rebal_daily, rebal_stats, event_log_df, drift_history


def compute_strategy_metrics(daily_values, initial_capital, benchmark_values=None):
    """Compute performance metrics from a daily portfolio value array or Series."""
    if isinstance(daily_values, pd.Series):
        daily_values = daily_values.values
    n = len(daily_values)
    if n < 2:
        return {'total_return': 0.0, 'cagr': 0.0, 'annualized_vol': 0.0, 'sharpe': 0.0,
                'max_drawdown': 0.0, 'skewness': 0.0, 'kurtosis': 0.0, 'avg_drawdown': 0.0,
                'tracking_error': 0.0, 'information_ratio': 0.0}
    total_return = daily_values[-1] / initial_capital - 1
    years = max(n - 1, 1) / 252.0
    cagr  = (daily_values[-1] / initial_capital) ** (1 / years) - 1 if years > 0 and daily_values[-1] > 0 else 0.0
    daily_rets = np.diff(daily_values) / daily_values[:-1]
    ann_vol    = float(np.std(daily_rets, ddof=1) * np.sqrt(252)) if len(daily_rets) > 1 else 0.0
    sharpe     = cagr / ann_vol if ann_vol > 0 else 0.0
    running_max = np.maximum.accumulate(daily_values)
    drawdowns   = (daily_values - running_max) / running_max
    max_dd      = float(np.min(drawdowns))
    avg_dd      = float(np.mean(drawdowns))
    skewness    = float(sp_stats.skew(daily_rets))   if len(daily_rets) > 2 else 0.0
    kurtosis    = float(sp_stats.kurtosis(daily_rets, fisher=True)) if len(daily_rets) > 3 else 0.0
    te = ir = 0.0
    if benchmark_values is not None:
        if isinstance(benchmark_values, pd.Series): benchmark_values = benchmark_values.values
        if len(benchmark_values) == n:
            bm_rets = np.diff(benchmark_values) / benchmark_values[:-1]
            active  = daily_rets - bm_rets
            te      = float(np.std(active, ddof=1) * np.sqrt(252)) if len(active) > 1 else 0.0
            if te > 1e-12:
                # FIX: IR numerator = CAGR difference (geometric), not mean(daily) * 252.
                # mean(daily) * 252 is an arithmetic approximation that diverges from realized
                # excess return over multi-year periods.
                bm_cagr = float((benchmark_values[-1] / benchmark_values[0]) ** (1.0 / years) - 1.0) \
                          if benchmark_values[0] > 0 else 0.0
                ir = (cagr - bm_cagr) / te
    return {'total_return': round(total_return, 6), 'cagr': round(cagr, 6),
            'annualized_vol': round(ann_vol, 6), 'sharpe': round(sharpe, 4),
            'max_drawdown': round(max_dd, 6), 'skewness': round(skewness, 4),
            'kurtosis': round(kurtosis, 4), 'avg_drawdown': round(avg_dd, 6),
            'tracking_error': round(te, 6), 'information_ratio': round(ir, 4)}


print(f'Repo root : {REPO_ROOT}')
print(f'Data dir  : {DATA_DIR}')
print('All engine functions loaded (no Streamlit dependency).')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — VALIDATE DATA FILES & LOAD PRICE / DIVIDEND / PROXY DATA
#
# Accepts either parquet or CSV for price data (parquet preferred if available).
# Auto-converts proxy_lookup.xlsx -> CSV if the CSV export is missing.
# ══════════════════════════════════════════════════════════════════════════════

PRICE_PARQUET = DATA_DIR / 'price_data.parquet'
PRICE_CSV     = DATA_DIR / 'price_data.csv'
DIV_PATH      = DATA_DIR / 'dividend_data.csv'
PROXY_CSV     = DATA_DIR / 'proxy_lookup.csv'

print(f'Repo root : {REPO_ROOT}')
print(f'Data dir  : {DATA_DIR}')
print()

# ── Resolve which price file to use ──────────────────────────────────────────
if PRICE_PARQUET.exists():
    PRICE_PATH = PRICE_PARQUET
elif PRICE_CSV.exists():
    PRICE_PATH = PRICE_CSV
else:
    PRICE_PATH = None

# ── Auto-convert proxy_lookup.xlsx -> CSV if needed ──────────────────────────
if not PROXY_CSV.exists():
    _xlsx_alt = DATA_DIR / 'proxy_lookup.xlsx'
    if _xlsx_alt.exists():
        print(f'Converting proxy_lookup.xlsx -> CSV ...')
        _tmp = pd.read_excel(_xlsx_alt)
        _tmp.to_csv(PROXY_CSV, index=False)
        print(f'  Created {PROXY_CSV.name}  ({len(_tmp):,} rows)')

# ── Pre-flight validation ────────────────────────────────────────────────────
_missing = []
if not DATA_DIR.exists():
    _missing.append(f'  DATA DIRECTORY not found: {DATA_DIR}')
if PRICE_PATH is None:
    _missing.append(f'  PRICE DATA not found: need price_data.parquet or price_data.csv in {DATA_DIR}')
if not PROXY_CSV.exists():
    _missing.append(f'  PROXY LOOKUP not found: {PROXY_CSV}'
                    f'\n         Need proxy_lookup.xlsx or the CSV export in {DATA_DIR}')
if not (REPO_ROOT / 'optimizer_msba_v1_engine.py').exists():
    _missing.append(f'  ENGINE MODULE not found: {REPO_ROOT / "optimizer_msba_v1_engine.py"}')

if _missing:
    print('=' * 70)
    print('MISSING FILES -- the notebook cannot run without these:\n')
    for m in _missing:
        print(m)
    if not DIV_PATH.exists():
        print(f'\n  (optional) dividend_data.csv not found: {DIV_PATH}')
        print(f'             Notebook will run without dividends.')
    print('\n' + '=' * 70)
    raise FileNotFoundError(
        f'{len(_missing)} required file(s) missing. See messages above.'
    )

print('All required files found.\n')

# ── Load price data ──────────────────────────────────────────────────────────
print(f'Loading price data from {PRICE_PATH.name} ...')
if PRICE_PATH.suffix == '.parquet':
    prices_raw = pd.read_parquet(PRICE_PATH)
else:
    prices_raw = pd.read_csv(PRICE_PATH)
prices_raw['PRICEDATE'] = pd.to_datetime(prices_raw['PRICEDATE'])
if 'TRADINGITEMSTATUSID' in prices_raw.columns:
    prices_raw = prices_raw[prices_raw['TRADINGITEMSTATUSID'].isin([1, 15])].copy()
prices_raw['PRICECLOSE'] = pd.to_numeric(prices_raw['PRICECLOSE'], errors='coerce')
prices_raw = prices_raw.dropna(subset=['PRICECLOSE'])
prices_raw['TICKERSYMBOL'] = prices_raw['TICKERSYMBOL'].astype(str).str.strip().str.upper()
print(f'Price data: {len(prices_raw):,} rows | {prices_raw["TICKERSYMBOL"].nunique()} tickers')
print(f'Date range: {prices_raw["PRICEDATE"].min().date()} \u2192 {prices_raw["PRICEDATE"].max().date()}')

# ── Load dividends (optional) ────────────────────────────────────────────────
# Price-only basis for cross-strategy comparability (matches run_backtest.py --full).
div_df = None
print('Running on a price-only basis (dividends off) for cross-strategy comparability.')

# ── Load and clean proxy lookup ──────────────────────────────────────────────
proxy_lookup_raw = pd.read_csv(PROXY_CSV)
proxy_lookup_raw['symbol']        = proxy_lookup_raw['symbol'].str.strip().str.upper()
proxy_lookup_raw['lookup_symbol'] = proxy_lookup_raw['lookup_symbol'].str.strip().str.upper()
PROXY_FULL = (
    proxy_lookup_raw[['symbol', 'lookup_type', 'lookup_symbol', 'order']]
    .drop_duplicates(subset=['symbol', 'lookup_symbol'])
    .sort_values(['symbol', 'order'])
    .reset_index(drop=True)
)
print(f'Proxy lookup loaded: {len(PROXY_FULL)} unique symbol\u2192proxy pairs')


def build_proxy_df(portfolio_tickers, start_date, end_date, prices_df):
    """
    Return a proxy_df filtered to:
      1. Only portfolio tickers that need proxies.
      2. Only proxy tickers that have price data in the given period.
    Falls back to the next-ranked proxy if the primary is unavailable.
    """
    avail_px = set(
        prices_df[
            (prices_df['PRICEDATE'] >= pd.Timestamp(start_date)) &
            (prices_df['PRICEDATE'] <= pd.Timestamp(end_date))
        ]['TICKERSYMBOL'].unique()
    )
    sub = PROXY_FULL[
        PROXY_FULL['symbol'].isin(portfolio_tickers) &
        PROXY_FULL['lookup_symbol'].isin(avail_px)
    ].copy()
    return sub.reset_index(drop=True)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — PORTFOLIO DEFINITIONS (auto-loaded from Models.xlsx)
#
# Loads ALL portfolios from Models.xlsx, using the latest snapshot per portfolio.
# Cash positions ("$") are removed and weights renormalized.
# Portfolios are classified into 3 categories by equity allocation:
#   Equity-Heavy (>=70%), Balanced (30-69%), Fixed Income-Heavy (<30%)
# ══════════════════════════════════════════════════════════════════════════════

MODELS_PATH = DATA_DIR / 'Models.xlsx'
if not MODELS_PATH.exists():
    raise FileNotFoundError(f'Models.xlsx not found: {MODELS_PATH}')

print('Loading portfolio definitions from Models.xlsx ...')
models_raw = pd.read_excel(MODELS_PATH)
models_raw['Trade Date'] = pd.to_datetime(models_raw['Trade Date'])
models_raw['Ticker'] = models_raw['Ticker'].astype(str).str.strip().str.upper()
models_raw['Weight'] = pd.to_numeric(models_raw['Weight'], errors='coerce').fillna(0.0)

# Use the LATEST snapshot per portfolio (most current weights)
latest_dates = models_raw.groupby('full_name')['Trade Date'].max().reset_index()
latest_dates.columns = ['full_name', 'Trade Date']
models_latest = models_raw.merge(latest_dates, on=['full_name', 'Trade Date'])

# Remove cash positions
models_latest = models_latest[models_latest['Ticker'] != '$'].copy()

# Equity asset classes (for categorization)
EQUITY_CLASSES = {
    'US Equities', 'International/Global Equities',
    'Emerging Market Equities', 'Global Equities',
}

# ── Build PORTFOLIOS dict and metadata ───────────────────────────────────────
PORTFOLIOS = {}
PORTFOLIO_META = {}  # name -> {model_family, portfolio_label, category, equity_pct}

for full_name, group in models_latest.groupby('full_name'):
    tickers = group['Ticker'].tolist()
    weights = group['Weight'].tolist()

    # Normalize weights to sum to 1.0 (handles cash removal + data errors like 9.0 weight)
    total_w = sum(weights)
    if total_w < 1e-6:
        continue  # skip empty portfolios
    weights = [w / total_w for w in weights]

    port_dict = {}
    for tk, w in zip(tickers, weights):
        if w > 1e-8:
            port_dict[tk] = port_dict.get(tk, 0.0) + w  # merge duplicate tickers

    if not port_dict:
        continue

    PORTFOLIOS[full_name] = port_dict

    # Classify by equity allocation
    equity_pct = group.loc[
        group['Asset Class'].isin(EQUITY_CLASSES), 'Weight'
    ].sum() / total_w

    if equity_pct >= 0.70:
        category = 'Equity-Heavy'
    elif equity_pct >= 0.30:
        category = 'Balanced'
    else:
        category = 'Fixed Income-Heavy'

    PORTFOLIO_META[full_name] = {
        'model_family': group['Model_Family'].iloc[0],
        'portfolio_label': group['Portfolio'].iloc[0],
        'category': category,
        'equity_pct': round(equity_pct, 3),
        'n_tickers': len(port_dict),
    }

# ── Summary ──────────────────────────────────────────────────────────────────
print(f'\nLoaded {len(PORTFOLIOS)} portfolios from {models_raw["Model_Family"].nunique()} model families')
print(f'Unique tickers across all portfolios: {len(set(tk for p in PORTFOLIOS.values() for tk in p))}')
print()

# Category breakdown
from collections import Counter
cat_counts = Counter(m['category'] for m in PORTFOLIO_META.values())
for cat in ['Equity-Heavy', 'Balanced', 'Fixed Income-Heavy']:
    count = cat_counts.get(cat, 0)
    families = sorted(set(m['model_family'] for n, m in PORTFOLIO_META.items() if m['category'] == cat))
    print(f'  {cat:20s}: {count:3d} portfolios  ({", ".join(families[:5])}{"..." if len(families) > 5 else ""})')

print(f'Ready: {len(PORTFOLIOS)} portfolios loaded')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — SIMULATION PARAMETERS
# ══════════════════════════════════════════════════════════════════════════════

# ── Time periods ──────────────────────────────────────────────────────────────
TIME_PERIODS = {
    'Bear (2007–2008)':     ('2007-01-01', '2008-12-31'),
    'Baseline (2010–2019)': ('2010-01-01', '2019-12-31'),
    'Bull (2023–2024)':     ('2023-01-01', '2024-12-31'),
    'Past 5Y (2021–2025)':  ('2021-01-01', '2025-12-31'),
    'Past 10Y (2016–2025)': ('2016-01-01', '2025-12-31'),
    'Past 20Y (2006–2025)': ('2006-01-01', '2025-12-31'),
}

# ── Capital & tax assumptions ─────────────────────────────────────────────────
INITIAL_CAPITAL = 1_000_000.0
TAX_RATES       = {'st_rate': 0.35, 'lt_rate': 0.20}
PRICE_FIELD     = 'PRICECLOSE'
COST_CONFIG     = {'commission_bps': 5.0, 'slippage_bps': 5.0, 'bid_ask_bps': 2.0}

# ── TLH threshold ─────────────────────────────────────────────────────────────
TLH_THRESHOLD = 0.10   # 10% loss triggers harvest

# ── Strategy grid ─────────────────────────────────────────────────────────────
# Calendar rebalancing frequencies
CALENDAR_FREQS = ['Monthly', 'Quarterly', 'Yearly']

# Absolute threshold bands (drift > X% triggers rebalance)
ABS_THRESHOLDS = [0.05, 0.10, 0.20]   # 5%, 10%, 20%

# Relative threshold bands (drift > X% of target weight triggers rebalance)
REL_THRESHOLDS = [0.25, 0.50]         # 25%, 50%

# ── Strategy labels ───────────────────────────────────────────────────────────
def strategy_label(strat_type, strat_val, tlh_on):
    """Build a human-readable label clearly showing all stacked dimensions."""
    tlh_tag = 'TLH=10%' if tlh_on else 'No TLH'
    if strat_type == 'calendar':
        return f'{strat_val} Rebal | {tlh_tag}'
    elif strat_type == 'abs':
        pct = int(strat_val * 100)
        return f'Abs Threshold {pct}% | {tlh_tag}'
    elif strat_type == 'rel':
        pct = int(strat_val * 100)
        return f'Rel Threshold {pct}% | {tlh_tag}'

print('Strategy grid:')
for freq in CALENDAR_FREQS:
    print(f'  Calendar: {strategy_label("calendar", freq, False)} / {strategy_label("calendar", freq, True)}')
for tol in ABS_THRESHOLDS:
    print(f'  Absolute: {strategy_label("abs", tol, False)} / {strategy_label("abs", tol, True)}')
for tol in REL_THRESHOLDS:
    print(f'  Relative: {strategy_label("rel", tol, False)} / {strategy_label("rel", tol, True)}')
print(f'\n{len(CALENDAR_FREQS) + len(ABS_THRESHOLDS) + len(REL_THRESHOLDS)} strategy types × 2 TLH states = '
      f'{(len(CALENDAR_FREQS) + len(ABS_THRESHOLDS) + len(REL_THRESHOLDS)) * 2} runs per combination')
print(f'{len(TIME_PERIODS)} periods × {len(PORTFOLIOS)} portfolios × '
      f'{(len(CALENDAR_FREQS) + len(ABS_THRESHOLDS) + len(REL_THRESHOLDS)) * 2} = '
      f'{len(TIME_PERIODS) * len(PORTFOLIOS) * (len(CALENDAR_FREQS) + len(ABS_THRESHOLDS) + len(REL_THRESHOLDS)) * 2} total runs')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — HELPER: BUILD WIDE PRICE MATRIX
# ══════════════════════════════════════════════════════════════════════════════

def build_prices_wide(prices_df, tickers, start_date, end_date, price_field='PRICECLOSE'):
    """Filter price data to tickers and date range, pivot to wide format."""
    mask = (
        prices_df['TICKERSYMBOL'].isin(tickers) &
        (prices_df['PRICEDATE'] >= pd.Timestamp(start_date)) &
        (prices_df['PRICEDATE'] <= pd.Timestamp(end_date))
    )
    subset = prices_df.loc[mask, ['TICKERSYMBOL', 'PRICEDATE', price_field]].copy()
    subset = subset.drop_duplicates(subset=['TICKERSYMBOL', 'PRICEDATE'])
    wide = subset.pivot(index='PRICEDATE', columns='TICKERSYMBOL', values=price_field)
    wide = wide.sort_index().dropna()
    return wide


def verify_ticker_coverage(portfolio_dict, start_date, end_date, period_name):
    """Check which tickers are present in price data for a given period."""
    tickers = list(portfolio_dict.keys())
    wide = build_prices_wide(prices_raw, tickers, start_date, end_date)
    present  = [t for t in tickers if t in wide.columns]
    missing  = [t for t in tickers if t not in wide.columns]
    coverage = sum(portfolio_dict[t] for t in present)
    print(f'  {period_name}: {len(present)}/{len(tickers)} tickers | weight coverage {coverage:.1%}')
    if missing:
        print(f'    Missing: {missing}')
    return present, missing


print('Ticker coverage verification:')
for port_name, port_dict in PORTFOLIOS.items():
    print(f'\n{port_name}:')
    for period_name, (start, end) in TIME_PERIODS.items():
        verify_ticker_coverage(port_dict, start, end, period_name)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — METRICS WRAPPER
# Wraps compute_strategy_metrics() to return title-case keys for consistency
# with the results DataFrame columns used in later cells.
# ══════════════════════════════════════════════════════════════════════════════

def compute_metrics(nav_series, initial_capital, benchmark_nav=None):
    """
    Compute full performance metrics for a NAV Series.
    Returns title-case keys matching the results DataFrame columns.
    """
    if nav_series is None or len(nav_series) < 2:
        return {}
    bm_vals = None
    if benchmark_nav is not None:
        bm_aligned = benchmark_nav.reindex(nav_series.index).ffill().dropna()
        common = nav_series.index.intersection(bm_aligned.index)
        if len(common) > 2:
            bm_vals = bm_aligned.loc[common].values
            nav_series = nav_series.loc[common]
    m = compute_strategy_metrics(nav_series, initial_capital, benchmark_values=bm_vals)
    return {
        'Final NAV ($)':     round(nav_series.iloc[-1], 2),
        'Total Return':      m['total_return'],
        'CAGR':              m['cagr'],
        'Volatility':        m['annualized_vol'],
        'Sharpe Ratio':      m['sharpe'],
        'Max Drawdown':      m['max_drawdown'],
        'Skewness':          m['skewness'],
        'Kurtosis':          m['kurtosis'],
        'Tracking Error':    m['tracking_error'],
        'Information Ratio': m['information_ratio'],
    }


print('compute_metrics wrapper ready.')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — MAIN SIMULATION LOOP  (V1 engine, parallelized)
#
# Phase 0: Build job list — pre-filter prices ONCE per period (not per portfolio)
# Phase 1: Run all simulations in parallel via joblib
# Phase 2: Pair TLH-on/off results, compute metrics (sequential, fast)
# ══════════════════════════════════════════════════════════════════════════════

import os
from time import time

try:
    from joblib import Parallel, delayed
    N_JOBS = max(1, os.cpu_count() - 1)
except ImportError:
    N_JOBS = 1

print(f'Parallelization: {N_JOBS} workers' if N_JOBS > 1 else 'Running sequentially (install joblib for parallel)')


def _run_sim_safe(repo_root_str, sim_params):
    """Worker: run one simulation inside a spawned process."""
    import sys
    if repo_root_str not in sys.path:
        sys.path.insert(0, repo_root_str)
    from optimizer_msba_v1_engine import run_optimizer_simulation as _run
    try:
        return {'ok': True, 'res': _run(**sim_params)}
    except Exception as e:
        return {'ok': False, 'error': str(e)}


# ── Phase 0: Build job list ──────────────────────────────────────────────────
t0 = time()
jobs = []

n_portfolios = len(PORTFOLIOS)
n_periods = len(TIME_PERIODS)
print(f'\nPhase 0: Building jobs for {n_portfolios} portfolios x {n_periods} periods ...')

# Pre-filter prices ONCE per period (avoid 948 full-DataFrame scans)
period_prices_cache = {}
for period_name, (start, end) in TIME_PERIODS.items():
    period_prices_cache[period_name] = prices_raw[
        (prices_raw['PRICEDATE'] >= pd.Timestamp(start)) &
        (prices_raw['PRICEDATE'] <= pd.Timestamp(end))
    ].copy()
    print(f'  Cached {period_name}: {len(period_prices_cache[period_name]):,} rows')

port_count = 0
for port_name, port_dict in PORTFOLIOS.items():
    port_count += 1
    tickers = list(port_dict.keys())

    for period_name, (start, end) in TIME_PERIODS.items():
        period_prices = period_prices_cache[period_name]

        # Build wide price matrix from period-cached prices
        subset = period_prices[period_prices['TICKERSYMBOL'].isin(tickers)].copy()
        if subset.empty:
            continue
        subset = subset.drop_duplicates(subset=['TICKERSYMBOL', 'PRICEDATE'])
        wide = subset.pivot(index='PRICEDATE', columns='TICKERSYMBOL', values=PRICE_FIELD)
        wide = wide.sort_index().dropna()
        avail_tickers = [t for t in tickers if t in wide.columns]

        raw_weights  = {t: port_dict[t] for t in avail_tickers}
        total_w      = sum(raw_weights.values())
        if total_w < 1e-6 or not avail_tickers:
            continue
        norm_weights = {t: w / total_w for t, w in raw_weights.items()}
        norm_w_list  = [norm_weights[t] for t in avail_tickers]

        period_proxy_df = build_proxy_df(avail_tickers, start, end, period_prices)

        # Pre-filter prices to only tickers this portfolio+proxies need
        all_needed = set(avail_tickers)
        if not period_proxy_df.empty:
            all_needed |= set(period_proxy_df['lookup_symbol'].unique())
        group_prices = period_prices[
            period_prices['TICKERSYMBOL'].isin(all_needed)
        ].copy()

        strategies = []
        for freq in CALENDAR_FREQS:
            strategies.append(('calendar', freq))
        for tol in ABS_THRESHOLDS:
            strategies.append(('abs', tol))
        for tol in REL_THRESHOLDS:
            strategies.append(('rel', tol))

        for strat_type, strat_val in strategies:
            strat_key = f'{strat_type}_{strat_val}'

            _trigger_dates = set()
            _threshold_rebal_count = 0
            if strat_type in ('abs', 'rel'):
                drift_mode = 'Absolute' if strat_type == 'abs' else 'Relative'
                if not wide.empty and len(avail_tickers) > 0:
                    _shares = {tk: INITIAL_CAPITAL * norm_weights[tk] / float(wide.iloc[0][tk])
                               for tk in avail_tickers if tk in wide.columns and wide.iloc[0][tk] > 0}
                    if _shares:
                        _cooldown = 0
                        for _i, _dt in enumerate(wide.index):
                            if _i == 0:
                                continue
                            _pt = {tk: float(wide.loc[_dt, tk]) for tk in _shares}
                            _tv = sum(_shares[tk] * _pt[tk] for tk in _shares)
                            if _tv <= 0:
                                continue
                            _cw = {tk: _shares[tk] * _pt[tk] / _tv for tk in _shares}
                            _drift = compute_drift(_cw, {tk: norm_weights[tk] for tk in _shares}, drift_mode)
                            if _cooldown <= 0 and any(_drift[tk] > strat_val for tk in _shares):
                                _trigger_dates.add(_dt)
                                _cooldown = 5
                                for tk in _shares:
                                    _shares[tk] = norm_weights[tk] * _tv / _pt[tk] if _pt[tk] > 0 else _shares[tk]
                            if _cooldown > 0:
                                _cooldown -= 1
                        _threshold_rebal_count = len(_trigger_dates & set(wide.index))

            for tlh_on in [False, True]:
                label = strategy_label(strat_type, strat_val, tlh_on)
                _tlh_thresh = TLH_THRESHOLD if tlh_on else 0.0

                params = dict(
                    prices_df=group_prices,
                    dividends_df=div_df,
                    tickers=avail_tickers,
                    weights=norm_w_list,
                    start_date=start,
                    end_date=end,
                    rebalance_frequency=strat_val if strat_type == 'calendar' else 'None',
                    tax_rates=TAX_RATES,
                    tlh_threshold=_tlh_thresh,
                    reinvest_dividends=True,
                    initial_capital=INITIAL_CAPITAL,
                    price_field=PRICE_FIELD,
                    static=False,
                    cost_config=COST_CONFIG,
                    proxy_df=period_proxy_df,
                    wash_sale_days=30,
                    compute_tax_alpha=tlh_on,
                )
                if strat_type != 'calendar':
                    params['forced_rebalance_dates'] = _trigger_dates

                meta = dict(
                    port_name=port_name, period_name=period_name,
                    strat_type=strat_type, strat_val=strat_val,
                    strat_key=strat_key, tlh_on=tlh_on, label=label,
                    threshold_rebal_count=_threshold_rebal_count,
                )
                jobs.append((params, meta))

    if port_count % 20 == 0 or port_count == n_portfolios:
        print(f'  {port_count}/{n_portfolios} portfolios prepared ({len(jobs):,} jobs so far) [{time()-t0:.0f}s]')

print(f'Phase 0 complete: {len(jobs):,} jobs in {time() - t0:.1f}s')

# ── Phase 1: Run simulations ─────────────────────────────────────────────────
print(f'\nPhase 1: Running {len(jobs):,} simulations ({N_JOBS} workers) ...')
t1 = time()
_repo_str = str(REPO_ROOT)

if N_JOBS > 1:
    raw_results = Parallel(n_jobs=N_JOBS, verbose=10)(
        delayed(_run_sim_safe)(_repo_str, p) for p, _ in jobs
    )
else:
    raw_results = []
    for i, (p, m) in enumerate(jobs):
        raw_results.append(_run_sim_safe(_repo_str, p))
        if (i + 1) % 50 == 0 or i == len(jobs) - 1:
            print(f'  {i + 1:,}/{len(jobs):,} done [{time()-t1:.0f}s]')

elapsed = time() - t1
print(f'Phase 1 complete: {elapsed:.1f}s  ({elapsed / max(len(jobs),1):.2f}s/run)')

# ── Phase 2: Assemble results ────────────────────────────────────────────────
print(f'\nPhase 2: Assembling results ...')
t2 = time()
all_results   = []
nav_store     = {}
failed_runs   = []

# First pass: collect TLH-off benchmarks for tracking error
benchmark_navs = {}
for (_, meta), raw in zip(jobs, raw_results):
    if raw['ok'] and not meta['tlh_on']:
        bm_key = (meta['port_name'], meta['period_name'], meta['strat_key'])
        benchmark_navs[bm_key] = raw['res']['nav_series']

# Second pass: build all result rows
for (_, meta), raw in zip(jobs, raw_results):
    run_id = (meta['port_name'], meta['period_name'], meta['label'])

    if not raw['ok']:
        failed_runs.append({'run_id': run_id, 'error': raw['error']})
        continue

    res = raw['res']
    nav_series = res['nav_series']
    nav_store[run_id] = nav_series

    _rdf = res.get('realized_df', pd.DataFrame())
    _tlh_events = int(_rdf['reason'].str.startswith('TLH_SELL').sum()) \
        if not _rdf.empty and 'reason' in _rdf.columns else 0
    _tlh_losses = _rebal_losses = 0.0
    if not _rdf.empty and 'reason' in _rdf.columns and 'gain_loss' in _rdf.columns:
        _is_tlh  = _rdf['reason'].str.startswith('TLH_SELL')
        _is_init = _rdf['reason'].str.startswith('INIT')
        _tlh_losses   = float(_rdf.loc[_is_tlh  & (_rdf['gain_loss'] < 0), 'gain_loss'].abs().sum())
        _rebal_losses = float(_rdf.loc[~_is_tlh & ~_is_init & (_rdf['gain_loss'] < 0), 'gain_loss'].abs().sum())

    extra_fields = {
        'Tax Paid ($)':              res.get('tax_paid_total', 0),
        'Losses Harvested ($)':      res.get('losses_harvested', 0),
        'TLH Losses ($)':            _tlh_losses,
        'Rebal Losses ($)':          _rebal_losses,
        'TLH Events':                _tlh_events,
        'Exec Costs ($)':            res.get('transaction_costs_total', 0),
        'Tax Alpha 2 ($)':           res.get('tax_alpha_2_final', None),
        'Loss CF ST ($)':            res.get('loss_carryforward_st', None),
        'Loss CF LT ($)':            res.get('loss_carryforward_lt', None),
        'Liquidation NAV ($)':       res.get('liquidation_nav', None),
        'Unrealized Gain ST ($)':    res.get('unrealized_gain_st', None),
        'Unrealized Gain LT ($)':    res.get('unrealized_gain_lt', None),
        'Liquidation Tax ($)':       res.get('liquidation_tax', None),
        'Liquidation Exec Cost ($)': res.get('liquidation_exec_cost', None),
    }
    if meta['strat_type'] in ('abs', 'rel'):
        extra_fields['Threshold Rebal Count'] = meta['threshold_rebal_count']

    bm_key = (meta['port_name'], meta['period_name'], meta['strat_key'])
    bm_nav = benchmark_navs.get(bm_key) if meta['tlh_on'] else None
    metrics = compute_metrics(nav_series, INITIAL_CAPITAL, benchmark_nav=bm_nav)

    _pm = PORTFOLIO_META.get(meta['port_name'], {})
    row = {
        'Portfolio':       meta['port_name'],
        'Model Family':    _pm.get('model_family', ''),
        'Category':        _pm.get('category', ''),
        'Equity %':        _pm.get('equity_pct', 0.0),
        'Period':          meta['period_name'],
        'Strategy':        meta['label'],
        'Strategy Type':   meta['strat_type'],
        'Strategy Value':  meta['strat_val'],
        'TLH':             'On (10%)' if meta['tlh_on'] else 'Off',
        'Rebal/Threshold': meta['strat_key'],
    }
    row.update(metrics)
    row.update(extra_fields)
    all_results.append(row)

# ── Summary ──────────────────────────────────────────────────────────────────
total_time = time() - t0
print(f'\n{"=" * 80}')
print(f'Complete: {len(all_results):,} succeeded, {len(failed_runs):,} failed')
print(f'Time: Phase 0={time()-t0-elapsed-(time()-t2):.0f}s  Phase 1={elapsed:.0f}s  Phase 2={time()-t2:.0f}s  Total={total_time:.0f}s')
if failed_runs:
    # Show summary, not all failures
    error_counts = {}
    for f in failed_runs:
        e = f['error'][:80]
        error_counts[e] = error_counts.get(e, 0) + 1
    print(f'\nFailure summary ({len(failed_runs):,} total):')
    for err, cnt in sorted(error_counts.items(), key=lambda x: -x[1])[:10]:
        print(f'  {cnt:4d}x  {err}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 — BUILD RESULTS DATAFRAME
# ══════════════════════════════════════════════════════════════════════════════

results_df = pd.DataFrame(all_results)

# Ordered columns for display
BASE_COLS = [
    'Portfolio', 'Period', 'Strategy', 'TLH',
    'Final NAV ($)', 'Total Return', 'CAGR', 'Volatility',
    'Sharpe Ratio', 'Max Drawdown', 'Tracking Error', 'Information Ratio',
    'Skewness', 'Kurtosis',
]
TAX_COLS = [
    'Tax Paid ($)', 'Losses Harvested ($)', 'TLH Events',
    'Exec Costs ($)', 'Tax Alpha 2 ($)', 'Loss CF ST ($)', 'Loss CF LT ($)',
]
display_cols = [c for c in BASE_COLS + TAX_COLS if c in results_df.columns]

print(f'Results table: {len(results_df)} rows × {len(results_df.columns)} columns')
print(f'Columns: {display_cols}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 9 — FULL RESULTS TABLE (per combination)
# ══════════════════════════════════════════════════════════════════════════════

fmt_pct  = lambda x: f'{x:.2%}' if pd.notna(x) else '—'
fmt_usd  = lambda x: f'${x:,.0f}' if pd.notna(x) else '—'
fmt_num  = lambda x: f'{x:.3f}' if pd.notna(x) else '—'

for (port, period), grp in results_df.groupby(['Portfolio', 'Period'], sort=False):
    # print(f'\n{'='*90}')
    # print(f'  Portfolio: {port}   |   Period: {period}')
    # print(f'{'='*90}')

    disp = grp[display_cols].copy()
    for col in ['Total Return', 'CAGR', 'Volatility', 'Max Drawdown', 'Tracking Error']:
        if col in disp.columns:
            disp[col] = disp[col].apply(fmt_pct)
    for col in ['Final NAV ($)', 'Tax Paid ($)', 'Losses Harvested ($)',
                'Exec Costs ($)', 'Tax Alpha 2 ($)', 'Loss CF ST ($)', 'Loss CF LT ($)']:
        if col in disp.columns:
            disp[col] = disp[col].apply(lambda x: fmt_usd(x) if pd.notna(x) else '—')
    for col in ['Sharpe Ratio', 'Information Ratio', 'Skewness', 'Kurtosis']:
        if col in disp.columns:
            disp[col] = disp[col].apply(fmt_num)

    # display(disp.drop(columns=['Portfolio', 'Period']).reset_index(drop=True))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 10 — TLH VALUE-ADD TABLE
# For each TLH run, compare to its no-TLH counterpart.
# Shows: CAGR delta, Tracking Error, Info Ratio, Tax Alpha 2.
# ══════════════════════════════════════════════════════════════════════════════

print('TLH VALUE-ADD vs No-TLH Benchmark (per strategy, period, portfolio)')
print('=' * 90)

tlh_on_df  = results_df[results_df['TLH'] == 'On (10%)'].copy()
tlh_off_df = results_df[results_df['TLH'] == 'Off'].copy()

merge_keys = ['Portfolio', 'Period', 'Rebal/Threshold']
merged = tlh_on_df.merge(
    tlh_off_df[merge_keys + ['CAGR', 'Sharpe Ratio', 'Max Drawdown', 'Final NAV ($)']],
    on=merge_keys,
    suffixes=('_TLH', '_BM'),
)

merged['CAGR Delta (TLH - BM)']   = merged['CAGR_TLH'] - merged['CAGR_BM']
merged['Sharpe Delta']             = merged['Sharpe Ratio_TLH'] - merged['Sharpe Ratio_BM']
merged['NAV Gain from TLH ($)']   = merged['Final NAV ($)_TLH'] - merged['Final NAV ($)_BM']

value_add_cols = [
    'Portfolio', 'Period', 'Strategy',
    'CAGR Delta (TLH - BM)', 'Tracking Error', 'Information Ratio',
    'Sharpe Delta', 'NAV Gain from TLH ($)', 'Tax Alpha 2 ($)',
    'Losses Harvested ($)', 'Tax Paid ($)', 'Exec Costs ($)',
]
va_disp = merged[[c for c in value_add_cols if c in merged.columns]].copy()

for col in ['CAGR Delta (TLH - BM)', 'Tracking Error', 'Information Ratio', 'Sharpe Delta']:
    if col in va_disp.columns:
        va_disp[col] = va_disp[col].apply(lambda x: f'{x:+.3f}' if pd.notna(x) else '—')
for col in ['NAV Gain from TLH ($)', 'Tax Alpha 2 ($)', 'Losses Harvested ($)',
            'Tax Paid ($)', 'Exec Costs ($)']:
    if col in va_disp.columns:
        va_disp[col] = va_disp[col].apply(lambda x: f'${x:+,.0f}' if pd.notna(x) else '—')

display(va_disp.reset_index(drop=True))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 13 — SUMMARY: BEST STRATEGY PER COMBINATION
# Ranks strategies within each portfolio × period by Sharpe, CAGR, and
# (for TLH-on runs) Information Ratio. Prints ranked top-3 per combo.
# ══════════════════════════════════════════════════════════════════════════════

print('TOP 3 STRATEGIES PER COMBINATION — RANKED BY SHARPE RATIO')
print('=' * 90)

for (port, period), grp in results_df.groupby(['Portfolio', 'Period'], sort=False):
    print(f'\n{'─'*90}')
    print(f'  {port}  |  {period}')
    print(f'{'─'*90}')

    # Best overall by Sharpe
    top3 = grp.nlargest(3, 'Sharpe Ratio')[[
        'Strategy', 'TLH', 'CAGR', 'Sharpe Ratio', 'Max Drawdown',
        'Tracking Error', 'Information Ratio',
    ]].copy()
    for col in ['CAGR', 'Max Drawdown', 'Tracking Error']:
        if col in top3.columns:
            top3[col] = top3[col].apply(lambda x: f'{x:.2%}' if pd.notna(x) else '—')
    for col in ['Sharpe Ratio', 'Information Ratio']:
        if col in top3.columns:
            top3[col] = top3[col].apply(lambda x: f'{x:.3f}' if pd.notna(x) else '—')
    print('  Top 3 by Sharpe Ratio (all strategies):')
    display(top3.reset_index(drop=True))

    # Best TLH-on by Info Ratio (TLH value add)
    tlh_grp = grp[grp['TLH'] == 'On (10%)'].dropna(subset=['Information Ratio'])
    if not tlh_grp.empty:
        best_tlh = tlh_grp.nlargest(3, 'Information Ratio')[[
            'Strategy', 'CAGR', 'Sharpe Ratio', 'Tracking Error', 'Information Ratio', 'Tax Alpha 2 ($)'
        ]].copy()
        for col in ['CAGR', 'Tracking Error']:
            if col in best_tlh.columns:
                best_tlh[col] = best_tlh[col].apply(lambda x: f'{x:.2%}' if pd.notna(x) else '—')
        for col in ['Sharpe Ratio', 'Information Ratio']:
            if col in best_tlh.columns:
                best_tlh[col] = best_tlh[col].apply(lambda x: f'{x:.3f}' if pd.notna(x) else '—')
        if 'Tax Alpha 2 ($)' in best_tlh.columns:
            best_tlh['Tax Alpha 2 ($)'] = best_tlh['Tax Alpha 2 ($)'].apply(
                lambda x: f'${x:+,.0f}' if pd.notna(x) else '—')
        print('  Top 3 TLH strategies by Information Ratio (TLH value-add):')
        display(best_tlh.reset_index(drop=True))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 12 — EXPORT TO CSV (grouped by category)
#
# Outputs:
#   full_results.csv           — all portfolios, all periods, all strategies
#   outputs/equity_heavy.csv   — portfolios with >=70% equity
#   outputs/balanced.csv       — portfolios with 30-69% equity
#   outputs/fixed_income.csv   — portfolios with <30% equity
# ══════════════════════════════════════════════════════════════════════════════

export_df = results_df.copy()

# ── Rename columns for clean export ──────────────────────────────────────────
export_df = export_df.rename(columns={
    'Period':               'Market State',
    'Strategy':             'Strategy Label',
    'Strategy Type':        'Rebal Type',
    'Strategy Value':       'Rebal Value',
    'TLH':                  'TLH Status',
    'Final NAV ($)':        'Final NAV',
    'Volatility':           'Volatility (Ann)',
    'Tracking Error':       'Tracking Error (Ann)',
    'Skewness':             'Return Skewness',
    'Kurtosis':             'Return Kurtosis',
    'Tax Paid ($)':         'Tax Paid',
    'Losses Harvested ($)': 'Realized Losses (All)',
    'TLH Losses ($)':       'TLH Losses',
    'Rebal Losses ($)':     'Rebal Losses',
    'TLH Events':           'TLH Event Count',
    'Exec Costs ($)':       'Execution Costs',
    'Tax Alpha 2 ($)':      'Tax Alpha 2',
    'Loss CF ST ($)':       'Loss Carryforward ST',
    'Loss CF LT ($)':       'Loss Carryforward LT',
    'Liquidation NAV ($)':    'Liquidation NAV',
    'Unrealized Gain ST ($)': 'Unrealized Gain ST',
    'Unrealized Gain LT ($)': 'Unrealized Gain LT',
    'Liquidation Tax ($)':    'Liquidation Tax',
    'Liquidation Exec Cost ($)': 'Liquidation Exec Cost',
    'Equity %':              'Equity Allocation',
})

# ── Map period names to market conditions ────────────────────────────────────
condition_map = {
    'Bear (2007\u20132008)':     'Bear Market',
    'Baseline (2010\u20132019)': 'Baseline Market',
    'Bull (2023\u20132024)':     'Bull Market',
    'Past 5Y (2021\u20132025)':  'Past 5 Years',
    'Past 10Y (2016\u20132025)': 'Past 10 Years',
    'Past 20Y (2006\u20132025)': 'Past 20 Years',
}
export_df['Market Condition'] = export_df['Market State'].map(condition_map).fillna(export_df['Market State'])

type_map = {'calendar': 'Calendar', 'abs': 'Absolute Threshold', 'rel': 'Relative Threshold'}
export_df['Rebal Type'] = export_df['Rebal Type'].map(type_map).fillna(export_df['Rebal Type'])

# ── Column ordering ──────────────────────────────────────────────────────────
ORDERED_COLS = [
    # Grouping
    'Category', 'Model Family', 'Portfolio', 'Equity Allocation',
    'Market Condition', 'Market State',
    'Rebal Type', 'Rebal Value', 'TLH Status', 'Strategy Label',
    # Performance
    'Final NAV', 'Total Return', 'CAGR',
    'Volatility (Ann)', 'Sharpe Ratio', 'Max Drawdown',
    'Return Skewness', 'Return Kurtosis',
    # Risk vs benchmark
    'Tracking Error (Ann)', 'Information Ratio',
    # Tax / TLH
    'Tax Paid', 'Realized Losses (All)', 'TLH Losses', 'Rebal Losses', 'TLH Event Count',
    'Execution Costs', 'Tax Alpha 2',
    'Loss Carryforward ST', 'Loss Carryforward LT',
    # Liquidation
    'Liquidation NAV', 'Unrealized Gain ST', 'Unrealized Gain LT',
    'Liquidation Tax', 'Liquidation Exec Cost',
]
final_cols = [c for c in ORDERED_COLS if c in export_df.columns]
export_df = export_df[final_cols]

# ── Sort: Category -> Model Family -> Portfolio -> Period -> Strategy ────────
_period_order = {
    'Bear Market': 0, 'Baseline Market': 1, 'Bull Market': 2,
    'Past 5 Years': 3, 'Past 10 Years': 4, 'Past 20 Years': 5,
}
export_df['_sort_period'] = export_df['Market Condition'].map(_period_order).fillna(9)
_cat_order = {'Equity-Heavy': 0, 'Balanced': 1, 'Fixed Income-Heavy': 2}
export_df['_sort_cat'] = export_df['Category'].map(_cat_order).fillna(9)
export_df = (
    export_df
    .sort_values(['_sort_cat', 'Model Family', 'Portfolio', '_sort_period', 'Rebal Type', 'TLH Status'])
    .drop(columns=['_sort_period', '_sort_cat'])
    .reset_index(drop=True)
)

# ── Export full results ──────────────────────────────────────────────────────
OUTPUTS_DIR = NOTEBOOK_DIR / 'outputs'
OUTPUTS_DIR.mkdir(exist_ok=True)

CSV_PATH = NOTEBOOK_DIR / 'full_results.csv'
export_df.to_csv(CSV_PATH, index=False, float_format='%.6f')
print(f'Exported: {CSV_PATH}')
print(f'Shape: {export_df.shape[0]} rows x {export_df.shape[1]} columns')

# ── Export per-category CSVs ─────────────────────────────────────────────────
for cat in ['Equity-Heavy', 'Balanced', 'Fixed Income-Heavy']:
    cat_df = export_df[export_df['Category'] == cat]
    if cat_df.empty:
        continue
    fname = cat.lower().replace('-', '_').replace(' ', '_') + '.csv'
    cat_path = OUTPUTS_DIR / fname
    cat_df.to_csv(cat_path, index=False, float_format='%.6f')
    n_ports = cat_df['Portfolio'].nunique()
    print(f'  {cat:20s}: {cat_path.name}  ({len(cat_df):,} rows, {n_ports} portfolios)')

print(f'\nColumn list ({len(final_cols)}):')
for i, col in enumerate(final_cols, 1):
    print(f'  {i:2d}. {col}')

display(export_df.head(3))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 13 — PLAYBOOK EXPORT (grouped by Category + Model Family)
# ══════════════════════════════════════════════════════════════════════════════
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

PLAYBOOK_PATH = NOTEBOOK_DIR / 'strategy_playbook.xlsx'
_period_order = {
    'Bear Market': 0, 'Baseline Market': 1, 'Bull Market': 2,
    'Past 5 Years': 3, 'Past 10 Years': 4, 'Past 20 Years': 5,
}

# ── 1. PLAYBOOK SHEET — TLH On rows, composite-ranked per portfolio x period ─
pb = export_df[export_df['TLH Status'] == 'On (10%)'].copy()

# Rank within each (Portfolio, Market Condition) group
_grp = ['Portfolio', 'Market Condition']
pb['_rank_sharpe']    = pb.groupby(_grp)['Sharpe Ratio'].rank(ascending=True)
pb['_rank_cagr']      = pb.groupby(_grp)['CAGR'].rank(ascending=True)
pb['_rank_dd']        = pb.groupby(_grp)['Max Drawdown'].rank(ascending=False)
pb['_ir_filled']      = pb['Information Ratio'].fillna(0)
pb['_rank_ir']        = pb.groupby(_grp)['_ir_filled'].rank(ascending=True)
pb['_ta_filled']      = pb['Tax Alpha 2'].fillna(0)
pb['_rank_tax_alpha'] = pb.groupby(_grp)['_ta_filled'].rank(ascending=True)

# Composite: Tax Alpha 30%, Sharpe 25%, CAGR 20%, MaxDD 15%, IR 10%
pb['Composite Score'] = (
    0.30 * pb['_rank_tax_alpha'] +
    0.25 * pb['_rank_sharpe'] +
    0.20 * pb['_rank_cagr'] +
    0.15 * pb['_rank_dd'] +
    0.10 * pb['_rank_ir']
).round(3)

pb['Rank'] = (
    pb.groupby(_grp)['Composite Score']
    .rank(ascending=False, method='min').astype(int)
)

pb['_sort'] = pb['Market Condition'].map(_period_order).fillna(9)
_cat_order = {'Equity-Heavy': 0, 'Balanced': 1, 'Fixed Income-Heavy': 2}
pb['_sort_cat'] = pb['Category'].map(_cat_order).fillna(9)
pb = pb.sort_values(['_sort_cat', 'Model Family', 'Portfolio', '_sort', 'Rank'])
pb = pb.drop(columns=[c for c in pb.columns if c.startswith('_')])

PLAYBOOK_COLS = [
    'Category', 'Model Family', 'Portfolio', 'Market Condition', 'Rank',
    'Rebal Type', 'Rebal Value', 'Strategy Label',
    'CAGR', 'Sharpe Ratio', 'Max Drawdown',
    'Realized Losses (All)', 'TLH Event Count', 'Tax Alpha 2',
    'Tracking Error (Ann)', 'Information Ratio', 'Composite Score',
]
pb_out = pb[[c for c in PLAYBOOK_COLS if c in pb.columns]].reset_index(drop=True)

for col in ['CAGR', 'Sharpe Ratio', 'Max Drawdown', 'Tracking Error (Ann)', 'Information Ratio']:
    if col in pb_out.columns:
        pb_out[col] = pb_out[col].round(4)
for col in ['Realized Losses (All)', 'Tax Alpha 2']:
    if col in pb_out.columns:
        pb_out[col] = pb_out[col].round(2)

# ── 2. TLH VALUE-ADD SHEET — delta metrics (On minus Off) ───────────────────
tlh_on  = export_df[export_df['TLH Status'] == 'On (10%)'].copy()
tlh_off = export_df[export_df['TLH Status'] == 'Off'].copy()

merge_keys = ['Category', 'Model Family', 'Portfolio', 'Market Condition', 'Rebal Type', 'Rebal Value']
va = tlh_on.merge(
    tlh_off[merge_keys + ['CAGR', 'Sharpe Ratio', 'Max Drawdown', 'Final NAV']],
    on=merge_keys,
    suffixes=('', '_bench'),
)
va['Delta CAGR']         = (va['CAGR']        - va['CAGR_bench']).round(4)
va['Delta Sharpe']       = (va['Sharpe Ratio'] - va['Sharpe Ratio_bench']).round(4)
va['Delta Max Drawdown'] = (va['Max Drawdown'] - va['Max Drawdown_bench']).round(4)
va['Delta Final NAV']    = (va['Final NAV']    - va['Final NAV_bench']).round(2)

va['_sort'] = va['Market Condition'].map(_period_order).fillna(9)
va['_sort_cat'] = va['Category'].map(_cat_order).fillna(9)
va = va.sort_values(['_sort_cat', 'Model Family', 'Portfolio', '_sort', 'Rebal Type'])

VA_COLS = [
    'Category', 'Model Family', 'Portfolio', 'Market Condition',
    'Rebal Type', 'Rebal Value', 'Strategy Label',
    'Delta CAGR', 'Delta Sharpe', 'Delta Max Drawdown', 'Delta Final NAV',
    'Realized Losses (All)', 'TLH Event Count', 'Tax Alpha 2',
    'Tracking Error (Ann)', 'Information Ratio',
]
va_out = va[[c for c in VA_COLS if c in va.columns]].reset_index(drop=True)

# ── 3. RECOMMENDATIONS SHEET — Rank 1 from each group ───────────────────────
rec_out = pb_out[pb_out['Rank'] == 1].copy()

REC_COLS = [
    'Category', 'Model Family', 'Portfolio', 'Market Condition',
    'Strategy Label', 'Rebal Type', 'Rebal Value',
    'CAGR', 'Sharpe Ratio', 'Max Drawdown',
    'Realized Losses (All)', 'Tax Alpha 2', 'Composite Score',
]
rec_out = rec_out[[c for c in REC_COLS if c in rec_out.columns]].reset_index(drop=True)

# ── 4. RAW DATA SHEET ────────────────────────────────────────────────────────
raw_out = export_df.copy()
raw_out['_sort'] = raw_out['Market Condition'].map(_period_order).fillna(9)
raw_out['_sort_cat'] = raw_out['Category'].map(_cat_order).fillna(9)
raw_out = (
    raw_out
    .sort_values(['_sort_cat', 'Model Family', 'Portfolio', '_sort', 'Rebal Type', 'TLH Status'])
    .drop(columns=['_sort', '_sort_cat'])
    .reset_index(drop=True)
)

# ── Write Excel ──────────────────────────────────────────────────────────────
with pd.ExcelWriter(PLAYBOOK_PATH, engine='openpyxl') as writer:
    rec_out.to_excel(writer, sheet_name='Recommendations', index=False)
    pb_out.to_excel(writer, sheet_name='Playbook',         index=False)
    va_out.to_excel(writer, sheet_name='TLH Value-Add',    index=False)
    raw_out.to_excel(writer, sheet_name='Raw Data',        index=False)

    # Auto-fit column widths
    for sheet_name in writer.sheets:
        ws = writer.sheets[sheet_name]
        for col_cells in ws.columns:
            max_len = max(len(str(c.value or '')) for c in col_cells)
            header_len = len(str(col_cells[0].value or ''))
            ws.column_dimensions[get_column_letter(col_cells[0].column)].width = min(max(max_len, header_len) + 2, 30)

print(f'Playbook exported -> {PLAYBOOK_PATH}')
print(f'  Recommendations : {len(rec_out):,} rows')
print(f'  Playbook        : {len(pb_out):,} rows')
print(f'  TLH Value-Add   : {len(va_out):,} rows')
print(f'  Raw Data        : {len(raw_out):,} rows')

display(rec_out.head(10))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 14 — STRATEGY RECOMMENDATION DASHBOARD
#
# Helps reviewers pick the best strategy by answering three questions:
#   1. What is your portfolio type?  (Equity-Heavy / Balanced / Fixed Income)
#   2. What market regime are you in? (Bear / Baseline / Bull / Long-horizon)
#   3. What is your objective?        (Max return / Min risk / Max TLH benefit / Best after-tax liquidation)
#
# Output: one recommendation table per (Category x Regime) showing the best
# strategy for each objective, with and without TLH.
# ══════════════════════════════════════════════════════════════════════════════

from IPython.display import display, Markdown

print('=' * 100)
print('  STRATEGY RECOMMENDATION DASHBOARD')
print('  Pick the best strategy by portfolio type, market regime, and objective')
print('=' * 100)

# Work from the renamed export_df (post Cell 12)
df = export_df.copy()

# ── 1. CATEGORY-LEVEL SUMMARY: Does TLH help or hurt? ───────────────────────
print('\n\n' + '=' * 100)
print('  SECTION 1 — TLH IMPACT BY CATEGORY AND MARKET REGIME')
print('  Average Tax Alpha 2 across all portfolios in each group')
print('=' * 100)

tlh_rows = df[df['TLH Status'] == 'On (10%)'].copy()
tlh_summary = (
    tlh_rows
    .groupby(['Category', 'Market Condition'])
    .agg(
        Portfolios=('Portfolio', 'nunique'),
        Avg_Tax_Alpha=('Tax Alpha 2', 'mean'),
        Median_Tax_Alpha=('Tax Alpha 2', 'median'),
        Pct_Positive=('Tax Alpha 2', lambda x: (x > 0).mean()),
        Avg_TLH_Events=('TLH Event Count', 'mean'),
    )
    .round(2)
)
tlh_summary.columns = ['Portfolios', 'Avg Tax Alpha ($)', 'Median Tax Alpha ($)',
                        '% Positive', 'Avg TLH Events']
tlh_summary['% Positive'] = (tlh_summary['% Positive'] * 100).round(1).astype(str) + '%'
display(Markdown('### TLH Impact Summary'))
display(tlh_summary)

# ── 2. BEST STRATEGY BY OBJECTIVE ───────────────────────────────────────────
print('\n\n' + '=' * 100)
print('  SECTION 2 — BEST STRATEGY PER OBJECTIVE')
print('  For each (Category x Regime), shows the top strategy for 4 objectives')
print('=' * 100)

objectives = {
    'Max Return (CAGR)':         ('CAGR',            True),   # highest CAGR
    'Min Risk (Sharpe)':         ('Sharpe Ratio',    True),   # highest Sharpe
    'Min Drawdown':              ('Max Drawdown',    False),  # least negative (closest to 0)
    'Best TLH Benefit':          ('Tax Alpha 2',     True),   # highest Tax Alpha
}

# For liquidation objective, only consider TLH-on rows
liq_obj = {
    'Best Liquidation Value':    ('Liquidation NAV', True),
}

for cat in ['Equity-Heavy', 'Balanced', 'Fixed Income-Heavy']:
    cat_df = df[df['Category'] == cat]
    if cat_df.empty:
        continue

    for regime in sorted(cat_df['Market Condition'].unique(),
                         key=lambda x: {'Bear Market': 0, 'Baseline Market': 1, 'Bull Market': 2,
                                        'Past 5 Years': 3, 'Past 10 Years': 4, 'Past 20 Years': 5}.get(x, 9)):
        regime_df = cat_df[cat_df['Market Condition'] == regime]
        if regime_df.empty:
            continue

        print(f'\n{"─" * 100}')
        print(f'  {cat}  |  {regime}  ({regime_df["Portfolio"].nunique()} portfolios)')
        print(f'{"─" * 100}')

        rows = []
        for obj_name, (col, ascending) in {**objectives, **liq_obj}.items():
            if col not in regime_df.columns:
                continue

            # For TLH-specific objectives, only look at TLH-on rows
            if col in ('Tax Alpha 2',):
                subset = regime_df[regime_df['TLH Status'] == 'On (10%)'].dropna(subset=[col])
            elif col == 'Liquidation NAV':
                subset = regime_df.dropna(subset=[col])
            else:
                subset = regime_df

            if subset.empty:
                continue

            if ascending:
                best = subset.loc[subset[col].idxmax()]
            else:
                best = subset.loc[subset[col].idxmax()]  # Max Drawdown: closest to 0 = max

            rows.append({
                'Objective': obj_name,
                'Strategy': best.get('Strategy Label', ''),
                'TLH': best.get('TLH Status', ''),
                'CAGR': f"{best.get('CAGR', 0):.2%}",
                'Sharpe': f"{best.get('Sharpe Ratio', 0):.3f}",
                'Max DD': f"{best.get('Max Drawdown', 0):.2%}",
                'Tax Alpha': f"${best.get('Tax Alpha 2', 0):+,.0f}" if pd.notna(best.get('Tax Alpha 2')) else 'N/A',
                'Liq NAV': f"${best.get('Liquidation NAV', 0):,.0f}" if pd.notna(best.get('Liquidation NAV')) else 'N/A',
                'Portfolio': best.get('Portfolio', '')[:40],
            })

        if rows:
            display(pd.DataFrame(rows).set_index('Objective'))

# ── 3. REBALANCING STRATEGY COMPARISON ───────────────────────────────────────
print('\n\n' + '=' * 100)
print('  SECTION 3 — REBALANCING STRATEGY COMPARISON')
print('  Average metrics by rebalancing type, across all portfolios (TLH On only)')
print('=' * 100)

rebal_summary = (
    tlh_rows
    .groupby(['Category', 'Rebal Type'])
    .agg(
        Avg_CAGR=('CAGR', 'mean'),
        Avg_Sharpe=('Sharpe Ratio', 'mean'),
        Avg_MaxDD=('Max Drawdown', 'mean'),
        Avg_Tax_Alpha=('Tax Alpha 2', 'mean'),
        Avg_Exec_Cost=('Execution Costs', 'mean'),
    )
    .round(4)
)
rebal_summary.columns = ['Avg CAGR', 'Avg Sharpe', 'Avg Max DD', 'Avg Tax Alpha ($)', 'Avg Exec Cost ($)']
for col in ['Avg CAGR', 'Avg Max DD']:
    rebal_summary[col] = rebal_summary[col].apply(lambda x: f'{x:.2%}')
rebal_summary['Avg Sharpe'] = rebal_summary['Avg Sharpe'].apply(lambda x: f'{x:.3f}')
rebal_summary['Avg Tax Alpha ($)'] = rebal_summary['Avg Tax Alpha ($)'].apply(lambda x: f'${x:+,.0f}')
rebal_summary['Avg Exec Cost ($)'] = rebal_summary['Avg Exec Cost ($)'].apply(lambda x: f'${x:,.0f}')

display(Markdown('### Average Metrics by Rebalancing Strategy (TLH On)'))
display(rebal_summary)

# ── 4. TLH ON vs OFF: WHEN IS TLH WORTH IT? ────────────────────────────────
print('\n\n' + '=' * 100)
print('  SECTION 4 — WHEN IS TLH WORTH IT?')
print('  For each category, shows % of scenarios where TLH improves each metric')
print('=' * 100)

on_df  = df[df['TLH Status'] == 'On (10%)'].copy()
off_df = df[df['TLH Status'] == 'Off'].copy()
merge_k = ['Category', 'Portfolio', 'Market Condition', 'Rebal Type', 'Rebal Value']
paired = on_df.merge(off_df[merge_k + ['CAGR', 'Sharpe Ratio', 'Final NAV', 'Liquidation NAV']],
                     on=merge_k, suffixes=('_on', '_off'))

worth_it = []
for cat in ['Equity-Heavy', 'Balanced', 'Fixed Income-Heavy']:
    cat_paired = paired[paired['Category'] == cat]
    if cat_paired.empty:
        continue
    n = len(cat_paired)
    worth_it.append({
        'Category': cat,
        'Scenarios': n,
        'Higher CAGR': f"{(cat_paired['CAGR_on'] > cat_paired['CAGR_off']).sum() / n:.0%}",
        'Higher Sharpe': f"{(cat_paired['Sharpe Ratio_on'] > cat_paired['Sharpe Ratio_off']).sum() / n:.0%}",
        'Higher Final NAV': f"{(cat_paired['Final NAV_on'] > cat_paired['Final NAV_off']).sum() / n:.0%}",
        'Higher Liq NAV': f"{(cat_paired['Liquidation NAV_on'] > cat_paired['Liquidation NAV_off']).sum() / n:.0%}",
    })

display(Markdown('### % of Scenarios Where TLH Outperforms No-TLH'))
display(pd.DataFrame(worth_it).set_index('Category'))

print('\n' + '=' * 100)
print('  END OF RECOMMENDATIONS')
print('=' * 100)